In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image


class SiameseNetwork(nn.Module):
    """
    Siamese Network for comparing RGB images (512x512)
    Uses a shared CNN encoder to extract features from both images
    """
    def __init__(self, embedding_dim=128):
        super(SiameseNetwork, self).__init__()
        
        # Shared feature extractor - relatively simple architecture
        self.encoder = nn.Sequential(
            # Block 1: 512x512x3 -> 256x256x32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 2: 256x256x32 -> 128x128x64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 3: 128x128x64 -> 64x64x128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 4: 64x64x128 -> 32x32x256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Global Average Pooling: 32x32x256 -> 1x1x256
            nn.AdaptiveAvgPool2d(1)
        )
        
        # Embedding layer to reduce dimensionality
        self.fc = nn.Sequential(
            nn.Linear(256, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )
        
    def forward_one(self, x):
        """Extract features for one image"""
        x = self.encoder(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        # L2 normalize embeddings
        x = F.normalize(x, p=2, dim=1)
        return x
    
    def forward(self, img1, img2):
        """Forward pass for both images"""
        embedding1 = self.forward_one(img1)
        embedding2 = self.forward_one(img2)
        return embedding1, embedding2


class ContrastiveLoss(nn.Module):
    """
    Contrastive Loss for Siamese Networks
    - Similar pairs (label=0): Minimize distance
    - Dissimilar pairs (label=1): Maximize distance up to margin
    
    Output: 0 for similar images, 1 for dissimilar images
    """
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
        
    def forward(self, embedding1, embedding2, label):
        """
        Args:
            embedding1: First embedding (batch_size, embedding_dim)
            embedding2: Second embedding (batch_size, embedding_dim)
            label: 0 for similar pairs, 1 for dissimilar pairs
        """
        # Euclidean distance
        euclidean_distance = F.pairwise_distance(embedding1, embedding2, p=2)
        
        # Contrastive loss
        loss_similar = (1 - label) * torch.pow(euclidean_distance, 2)
        loss_dissimilar = label * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2)
        
        loss = torch.mean(loss_similar + loss_dissimilar)
        return loss


class SiameseDataset(Dataset):
    """
    Custom Dataset for Siamese Network
    Returns pairs of images with labels (0=similar, 1=dissimilar)
    """
    def __init__(self, image_pairs, labels, transform=None):
        """
        Args:
            image_pairs: List of tuples [(img1_path, img2_path), ...]
            labels: List of labels [0, 1, 0, ...] where 0=similar, 1=dissimilar
            transform: Image transformations
        """
        self.image_pairs = image_pairs
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_pairs)
    
    def __getitem__(self, idx):
        img1_path, img2_path = self.image_pairs[idx]
        label = self.labels[idx]
        
        # Load images
        img1 = Image.open(img1_path).convert('RGB')
        img2 = Image.open(img2_path).convert('RGB')
        
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
            
        return img1, img2, torch.tensor(label, dtype=torch.float32)


def get_transforms(train=True):
    """
    Get image transformations
    For training: Add augmentation (rotation, zoom)
    For validation: Only normalize
    """
    if train:
        return transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.RandomRotation(15),  # Random rotation
            transforms.RandomResizedCrop(512, scale=(0.8, 1.0)),  # Random zoom
            transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Color variation
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])


def compute_similarity(model, img1, img2, device='cuda'):
    """
    Compute similarity between two images
    Returns: similarity score (0=similar, 1=dissimilar)
    """
    model.eval()
    with torch.no_grad():
        img1 = img1.unsqueeze(0).to(device)
        img2 = img2.unsqueeze(0).to(device)
        
        embedding1, embedding2 = model(img1, img2)
        distance = F.pairwise_distance(embedding1, embedding2, p=2)
        
        # Normalize distance to [0, 1] range
        # Assuming max distance is around sqrt(2) for normalized embeddings
        similarity = torch.clamp(distance / 1.414, 0, 1)
        
    return similarity.item()


# Training function
def train_epoch(model, dataloader, criterion, optimizer, device='cuda'):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    for batch_idx, (img1, img2, labels) in enumerate(dataloader):
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
        
        # Forward pass
        embedding1, embedding2 = model(img1, img2)
        loss = criterion(embedding1, embedding2, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % 10 == 0:
            print(f'Batch [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}')
    
    return total_loss / len(dataloader)


# Validation function
def validate(model, dataloader, criterion, device='cuda'):
    """Validate the model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for img1, img2, labels in dataloader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            
            embedding1, embedding2 = model(img1, img2)
            loss = criterion(embedding1, embedding2, labels)
            total_loss += loss.item()
            
            # Calculate accuracy (threshold at 0.5)
            distances = F.pairwise_distance(embedding1, embedding2, p=2)
            predictions = (distances > 0.7).float()  # 0.7 is threshold
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100 * correct / total
    return total_loss / len(dataloader), accuracy


# Example usage
if __name__ == "__main__":
    # Initialize model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = SiameseNetwork(embedding_dim=128).to(device)
    
    # Loss and optimizer
    criterion = ContrastiveLoss(margin=1.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    
    print("Siamese Network initialized successfully!")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Device: {device}")
    
    # Example: Create dummy dataset (replace with your actual data)
    # Format: [(img1_path, img2_path), ...] and [label, ...]
    # where label=0 for similar pairs, label=1 for dissimilar pairs
    
    """
    # Example training loop structure:
    
    train_pairs = [
        ('path/to/img1.jpg', 'path/to/img1_rotated.jpg'),  # Similar pair
        ('path/to/img1.jpg', 'path/to/img2.jpg'),          # Dissimilar pair
        # ... more pairs
    ]
    train_labels = [0, 1, ...]  # 0=similar, 1=dissimilar
    
    train_dataset = SiameseDataset(train_pairs, train_labels, transform=get_transforms(train=True))
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
    
    val_dataset = SiameseDataset(val_pairs, val_labels, transform=get_transforms(train=False))
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=4)
    
    # Training loop
    num_epochs = 50
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        print(f'\nEpoch [{epoch+1}/{num_epochs}]')
        
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        scheduler.step(val_loss)
        
        print(f'Train Loss: {train_loss:.4f}')
        print(f'Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.2f}%')
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_siamese_model.pth')
            print('Model saved!')
    """

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
import os
import random


class ContrastiveLoss(nn.Module):
    """
    Contrastive loss function for Siamese networks.
    """
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
        
    def forward(self, output1, output2, label):
        """
        Args:
            output1: First embedding vector
            output2: Second embedding vector
            label: 1 if same sample, 0 if different samples
        """
        euclidean_distance = F.pairwise_distance(output1, output2)
        loss = torch.mean((1 - label) * torch.pow(euclidean_distance, 2) +
                         label * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))
        return loss


class SiameseNetwork(nn.Module):
    """
    Siamese Network for DHM reconstructed phase images.
    Processes 512x512 RGB images.
    """
    def __init__(self, embedding_dim=256):
        super(SiameseNetwork, self).__init__()
        
        # Shared CNN backbone
        self.encoder = nn.Sequential(
            # Block 1: 512x512 -> 256x256
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 2: 256x256 -> 128x128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 3: 128x128 -> 64x64
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 4: 64x64 -> 32x32
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 5: 32x32 -> 16x16
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        
        # Global average pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layers for embedding
        self.fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, embedding_dim)
        )
        
    def forward_one(self, x):
        """Forward pass for one branch."""
        x = self.encoder(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
    
    def forward(self, input1, input2):
        """Forward pass for both branches."""
        output1 = self.forward_one(input1)
        output2 = self.forward_one(input2)
        return output1, output2




def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (img1, img2, label) in enumerate(dataloader):
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        output1, output2 = model(img1, img2)
        
        # Calculate loss
        loss = criterion(output1, output2, label)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Calculate accuracy based on distance threshold
        euclidean_distance = F.pairwise_distance(output1, output2)
        predictions = (euclidean_distance > 0.5).long()
        correct += (predictions == label).sum().item()
        total += label.size(0)
        
        if (batch_idx + 1) % 10 == 0:
            print(f'Batch [{batch_idx+1}/{len(dataloader)}], '
                  f'Loss: {loss.item():.4f}, '
                  f'Acc: {100.*correct/total:.2f}%')
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for img1, img2, label in dataloader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            
            output1, output2 = model(img1, img2)
            loss = criterion(output1, output2, label)
            
            running_loss += loss.item()
            
            euclidean_distance = F.pairwise_distance(output1, output2)
            predictions = (euclidean_distance > 0.5).long()
            correct += (predictions == label).sum().item()
            total += label.size(0)
    
    val_loss = running_loss / len(dataloader)
    val_acc = 100. * correct / total
    
    return val_loss, val_acc


def main():
    """Main training function."""
    # Hyperparameters
    BATCH_SIZE = 16
    EMBEDDING_DIM = 256
    LEARNING_RATE = 0.0001
    NUM_EPOCHS = 50
    MARGIN = 1.0
    
    # Paths (modify these to your actual paths)
    IMAGE_DIR = '/path/to/reconstructed/phase/images'
    UNET_DIR = '/path/to/unet/outputs'  # Optional
    ORIGINAL_DIR = '/path/to/original/images'  # Optional
    
    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    # Create datasets
    train_dataset = DHMSiameseDataset(
        image_dir=IMAGE_DIR,
        unet_dir=UNET_DIR,
        original_dir=ORIGINAL_DIR,
        mode='train'
    )
    
    val_dataset = DHMSiameseDataset(
        image_dir=IMAGE_DIR,
        unet_dir=UNET_DIR,
        original_dir=ORIGINAL_DIR,
        mode='val'
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    
    # Initialize model
    # Choose between custom or ResNet-based
    model = SiameseNetwork(embedding_dim=EMBEDDING_DIM).to(device)
    # model = ResNetSiameseNetwork(embedding_dim=EMBEDDING_DIM).to(device)
    
    # Loss and optimizer
    criterion = ContrastiveLoss(margin=MARGIN)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    
    # Training loop
    best_val_loss = float('inf')
    
    for epoch in range(NUM_EPOCHS):
        print(f'\nEpoch [{epoch+1}/{NUM_EPOCHS}]')
        print('-' * 50)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        
        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_acc': val_acc,
            }, 'best_siamese_model.pth')
            print(f'Saved best model with val_loss: {val_loss:.4f}')
    
    print('\nTraining completed!')


if __name__ == '__main__':
    main()